# LangChain 기반 LLM 활용 및 프롬프트 엔지니어링

- LLM API 가 **상태 없는 텍스트 생성기**임을 이해하고, 대화 메모리가 어떻게 만들어지는지 설명한다
- ChatModel · PromptTemplate · OutputParser 를 LCEL 로 연결할 수 있다
- Few-Shot · CoT · Structured Output 을 **언제 쓰는지** 판단 기준을 갖는다

---

## 0. 실행 준비

이 노트북은 `.env` 의 `OPENAI_API_KEY` 를 씁니다. **키가 없으면 조용히 넘어가지 않고 여기서 멈춥니다** —
뒤쪽 셀에서 알 수 없는 에러로 죽는 것보다 여기서 분명히 실패하는 편이 낫습니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), (
    "OPENAI_API_KEY 가 없습니다. 프로젝트 루트에 .env 를 만들고 키를 넣으세요."
)
print("준비 완료")

---

## 1. LLM API 의 본질

먼저 무엇을 다루고 있는지부터 정확히 합니다.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

resp = llm.invoke([
    SystemMessage("당신은 사내 인사 담당자입니다. 규정에 근거해 간결히 답하세요."),
    HumanMessage("연차는 어떻게 신청하나요?"),
])
print(resp.content)
print("\n입력 토큰:", resp.usage_metadata["input_tokens"])

---

## 2. 시연 D1 — 호출 횟수와 토큰 증가

---

## 🔮 예측 — D1

**같은 체인을 `invoke` 로 부를 때와 `stream` 으로 부를 때, **모델 API 호출 횟수**가 다를까요?**

<details>
<summary>이 질문을 하는 이유</summary>

> 스트리밍을 '다른 API' 로 오해하는 경우가 많습니다. 실제로는 같은 호출의 수신 방식만 다릅니다.

</details>

**대조 실행** — 아래 두 셀을 연달아 실행합니다.

- **A:** `invoke` — 완성된 답을 한 번에 받는다
- **B:** `stream` — 같은 호출을 토큰 단위로 나눠 받는다

In [ ]:
# A: invoke
resp = llm.invoke("사내 보안 정책을 한 문장으로 요약해줘.")
print("[invoke]", resp.content)
print("입력 토큰:", resp.usage_metadata["input_tokens"],
      "· 출력 토큰:", resp.usage_metadata["output_tokens"])

In [ ]:
# B: stream 호출 — 1회 API 호출로 응답 청크를 스트리밍 수신
print("[stream] ", end="")
chunks = 0
for chunk in llm.stream("사내 보안 정책을 한 문장으로 요약해줘."):
    print(chunk.content, end="", flush=True)
    chunks += 1
print(f"\n\n조각 {chunks}개로 나뉘어 도착 — 그러나 모델 호출은 1회다.")

> ### ▶ 함께 실행 — D1
>
> **`invoke` 와 `stream` 을 각각 돌리고, 이어서 3턴 대화를 실행합니다.**
>
> 위 셀을 여러분 노트북에서도 실행해 보세요.
>
> 🔍 **여러분 화면에서 볼 것** — 턴이 늘수록 **입력 토큰이 커지는가**. 호출 수는 1회로 같습니다 — 토큰 절대값은 서로 다를 수 있습니다

---

### 📊 결과 해석

**예상되는 결과:** 두 방식 모두 모델 호출은 1회. `stream` 은 같은 응답을 조각으로 받을 뿐입니다.

| 결과 | 해설 및 원인 분석 |
|---|---|
| 예상대로 나옴 | 스트리밍은 별도 API 가 아니라 **수신 방식**입니다. 비용도 같습니다. |
| 반대로 나옴 | 조각 수가 1이면 응답이 짧아서입니다. 더 긴 질문으로 재실행하세요. |
| 차이가 없음 | 핵심은 조각 수가 아니라 **호출이 1회**라는 점입니다. |

### 💡 대화를 3턴 이어갈 때 입력 토큰의 변화

다음 셀은 대화를 누적하면서 **매 턴 입력 토큰을 기록**합니다. 실행 전에 결과를 예측해 보세요:
*"3번째 턴의 입력 토큰은 1번째 턴의 몇 배가 될까요?"*

In [ ]:
history = []
curve = []

for turn in ["연차는 며칠인가요?", "반차도 되나요?", "이월은 가능한가요?"]:
    history.append(HumanMessage(turn))
    resp = llm.invoke(history)
    history.append(resp)                      # AI 응답도 다음 턴의 대화 이력(입력 토큰)으로 누적됨
    curve.append(resp.usage_metadata["input_tokens"])
    print(f"턴 {len(curve)} | 입력 토큰 {resp.usage_metadata['input_tokens']:>5} | {turn}")

print(f"\n입력 토큰 곡선: {curve}")
print(f"3턴째는 1턴째의 {curve[-1] / curve[0]:.1f}배")

---

### 📊 결과 해석

**예상되는 결과:** 입력 토큰이 계단식으로 증가합니다. (기준 출력: `[16, 218, 401]`, 3번째 턴에서 약 25배 증가) 절댓값은 응답 길이에 따라 실행마다 다를 수 있으나, 핵심은 배수 자체가 아니라 단조 증가 추세와 자릿수의 변화입니다.

| 결과 상황 | 해설 및 원인 분석 |
|---|---|
| 예상대로 증가함 | 모델이 대화를 스스로 기억하는 것이 아니라, **우리가 이전 대화 이력을 매번 다시 전송**하고 있는 것입니다. |
| 증가하지 않음 | 토큰 수가 증가하지 않았다면 history 누적 로직이 누락된 것입니다. 코드를 다시 확인해 보세요. |
| 증가 폭이 미미함 | 질문 내용이 짧은 경우 발생할 수 있으며, 절댓값보다는 **단조 증가 추세**를 확인하는 것이 핵심입니다. |

> 🎯 **핵심** — 모델은 자체적으로 대화 상태를 기억하지 않습니다. 매 호출마다 이전 대화 이력을 모두 다시 전송해야 하므로, 대화가 길어질수록 **호출 1건당 비용이 지속적으로 증가**합니다.

| 흔한 오해 | 실제 |
|---|---|
| 모델이 대화를 기억한다 | 매 호출마다 전체 이력을 재전송합니다. 대화 메모리는 애플리케이션이 직접 관리합니다 |

---

## 3. LangChain LCEL

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 사내 {role} 담당자입니다. 3문장 이내로 답하세요."),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

print(chain.invoke({"role": "인사", "question": "연차 이월 규정을 알려줘"}))

In [ ]:
# 파서(StrOutputParser) 적용 전후의 토큰 메타데이터(usage_metadata) 유지 여부 비교
raw = (prompt | llm).invoke({"role": "인사", "question": "연차 이월 규정을 알려줘"})
print("파서 앞 — 타입:", type(raw).__name__, "· 토큰:", raw.usage_metadata["total_tokens"])

parsed = (prompt | llm | StrOutputParser()).invoke(
    {"role": "인사", "question": "연차 이월 규정을 알려줘"})
print("파서 뒤 — 타입:", type(parsed).__name__, "· usage_metadata 접근 가능?",
      hasattr(parsed, "usage_metadata"))

---

## 4. 프롬프트 엔지니어링 —  출력 형식 비교

---

## 🔮 예측 — D2

**같은 문의를 ① 기본(Prompt만 사용) ② Few-Shot ③ Structured Output 으로 분류시킵니다.
**이 중 어떤 결과를 후속 코드에서 바로 활용할 수 있을까요?****


<details>
<summary>이 질문을 하는 이유</summary>

> "프롬프트만 잘 작성하면 출력 형식도 알아서 맞춰진다"는 흔한 오해를 바로잡기 위함입니다.

</details>

**대조 실행** — 아래 두 셀을 연달아 실행합니다.

- **A:** 기본 프롬프트 / Few-Shot — 사람이 읽기엔 자연스러우나 파싱이 불안정함
- **B:** Structured Output — 프로그램 코드가 안정적으로 직접 참조 가능

In [ ]:
INQUIRY = "지난달 결제가 두 번 청구된 것 같습니다. 확인 부탁드립니다."

# A-1: 기본 프롬프트 (Plain)
plain = llm.invoke(f"다음 고객 문의를 분류하세요: {INQUIRY}")
print("[기본 프롬프트]\n", plain.content, "\n")

# A-2: Few-Shot — 형식을 예시로 보여준다
few_shot = llm.invoke(f"""다음 고객 문의를 분류하세요.

예시:
문의: 로그인이 안 됩니다 → 계정
문의: 환불 언제 되나요 → 결제
문의: 사용법을 모르겠어요 → 문의

문의: {INQUIRY} →""")
print("[Few-Shot]\n", few_shot.content)

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class Triage(BaseModel):
    """고객 문의 분류 결과."""
    category: Literal["계정", "결제", "문의", "기타"] = Field(description="문의 유형")
    urgent: bool = Field(description="즉시 대응이 필요한가")
    summary: str = Field(description="한 문장 요약")

# B: Structured Output — 반환값이 곧 파이썬 객체다
result = llm.with_structured_output(Triage).invoke(f"고객 문의를 분류하세요: {INQUIRY}")

print("[Structured Output]", result, "\n")
print("→ 다음 코드가 바로 쓸 수 있다:")
print(f"   result.category = {result.category!r}")
print(f"   result.urgent   = {result.urgent!r}")

# Pydantic 모델을 통한 속성 직접 접근 및 안전한 조건 분기
if result.urgent:
    print("\n   → 긴급 큐로 라우팅")
else:
    print("\n   → 일반 큐로 라우팅")

> ### ▶ 함께 실행 — D2
>
> **세 가지 방식을 차례로 실행합니다.**
>
> 위 셀을 여러분 노트북에서도 실행해 보세요.
>
> 🔍 **여러분 화면에서 볼 것** — Structured Output 만 `.category` 로 바로 꺼내지는가. **3회 반복의 변동 폭은 실행 환경마다 다를 수 있습니다**
>
> ⚠️ 실행 결과가 예시와 **다를 수 있습니다.** 이는 오류가 아니며, 생성 모델의 통계적 변동성을 확인하는 과정입니다.

---

### 📊 결과 해석

**예상되는 결과:** ①, ②는 문자열(텍스트) 형태로 반환되어 별도의 파싱이 필요하지만, ③은 Pydantic 객체로 반환되어 `.category` 속성으로 즉시 접근할 수 있습니다.

| 결과 상황 | 해설 및 원인 분석 |
|---|---|
| 예상대로 출력됨 | ①, ②도 응답 내용은 맞을 수 있습니다. 문제는 **후속 코드가 안정적으로 값을 꺼내 쓰기 어렵다**는 점입니다. |
| Few-Shot도 정형화된 경우 | "이번 실행에서는 우연히 형식이 맞았을 수 있습니다. 하지만 **다음 실행에서도 일관되게 보장**될 수 있을까요?" |
| 결과 차이가 모호한 경우 | 단순 출력 텍스트가 비슷해 보이더라도 데이터의 **반환 타입(`type`)** 을 확인해 보세요. `str`과 `Triage` 객체의 차이가 핵심입니다. |

### 💡 심화 검증 — `temperature=0` 설정 시 결과는 항상 동일할까요?

실행 전에 결과를 예상해 보세요.

In [ ]:
outs = [llm.invoke("사내 보안 정책을 한 문장으로 요약해줘.").content for _ in range(3)]
for i, o in enumerate(outs, 1):
    print(f"{i}회차: {o[:60]}…")
print(f"\n서로 다른 응답: {len(set(outs))}종 / 3회")

---

### 📊 결과 해석

**예상되는 결과:** 3회 반복 실행 시 2~3종의 서로 다른 응답이 생성될 수 있습니다. (실제 테스트 환경에서도 실행 회차마다 일부 차이가 발생함)

| 결과 상황 | 해설 및 원인 분석 |
|---|---|
| 결과가 다르게 나온 경우 | "`temperature=0`은 **샘플링 확률 분포만** 고정할 뿐, 완전한 결정론적 재현을 보장하는 것은 아닙니다." |
| 모두 동일하게 나온 경우 | "이번 실행에서는 결과가 같았으나, 이는 **경향성일 뿐 절대적인 보장**은 아닙니다. (항상 동일하다고 단정하면 안 됨)" |
| 차이가 모호한 경우 | '대체로 유사한 경향'과 '항상 완벽히 일치함(결정론)'의 개념적 차이를 명확히 짚어줍니다. |

---

## 5. 미니실습 M1 — 예시 개수를 바꿔 본다

---

### 🔧 미니실습 M1 — Few-Shot 예시를 3개에서 1개로 줄이기

**변경할 부분**: `USE = EXAMPLES[:3]` 의 슬라이싱 숫자 변경

**확인할 항목**:
① **입력 토큰이 얼마나 감소했는가** (결정론적 수치로 모든 환경에서 동일함)
② **분류 결과의 일관성이 유지되는가** (관찰 대상)

> 3~5분간 실습을 진행합니다. 해결이 어려운 경우 **하단의 정답 셀을 실행**하여 다음 단계로 넘어가셔도 좋습니다.

In [ ]:
EXAMPLES = [
    ("배송이 3일째 안 왔습니다", "배송"),
    ("환불 절차가 궁금합니다", "환불"),
    ("앱이 로그인 화면에서 멈춥니다", "기술지원"),
]

# TODO — 예시를 3개에서 1개로 줄여 보세요.  힌트: EXAMPLES[:1]
USE = EXAMPLES[:3]

shots = "\n".join(f"- {q} → {a}" for q, a in USE)
resp = llm.invoke(
    f"다음 고객 문의를 분류하세요. 카테고리 한 단어만 답하세요.\n"
    f"[예시]\n{shots}\n\n[문의] {INQUIRY}"
)
print(f"예시 {len(USE)}개 · 입력 토큰 {resp.usage_metadata['input_tokens']} · 분류 {resp.content.strip()[:20]!r}")

<details>
<summary>정답 — M1</summary>

아래 셀이 기준 구현입니다. 직접 푼 결과와 비교해 보세요.

</details>

In [ ]:
# 두 경우를 나란히 돌려 봅니다.
for n in (3, 1):
    USE = EXAMPLES[:n]
    shots = "\n".join(f"- {q} → {a}" for q, a in USE)
    resp = llm.invoke(
        f"다음 고객 문의를 분류하세요. 카테고리 한 단어만 답하세요.\n"
        f"[예시]\n{shots}\n\n[문의] {INQUIRY}"
    )
    print(f"예시 {n}개 · 입력 토큰 {resp.usage_metadata['input_tokens']:>4} · "
          f"분류 {resp.content.strip()[:20]!r}")


> 🎯 **핵심** — 프롬프트의 모든 글자는 **매 호출마다 비용**입니다.
> 예시를 늘리는 결정은 "품질이 오르는가"만이 아니라 **"오른 만큼 값을 하는가"** 로 해야 합니다.

---

## 정리

- LLM 은 상태가 없다. 대화 메모리는 재전송으로 만들어지고, 그래서 비용이 누적된다
- LCEL 의 각 경계는 확장점이자 관측점이다. **토큰은 파서 앞에서** 잡는다
- Structured Output 이 있어야 다음 코드가 분기할 수 있다